# Data Cleaning – Curia Vista (completed "Anfragen")

Minimal cleaning notebook covering the mandatory steps from the project description:

1. Check for gaps / missing data
2. Check datatypes, change if needed
3. Check if values lie in the expected range
4. Identify outliers, treat them reasonably
5. Format the dataset for the task (combine, merge, resample, …)
6. Enrich the dataset with at least one additional column

**Input:** `Collection/data/curia_vista_anfragen_erledigt.csv` (output of `scrape_curia_vista.py`)
**Output:** `Preparation/data/`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_FILE = Path("../Collection/data/curia_vista_anfragen_erledigt.csv")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

# business_number must be read as string, otherwise "26.1030" becomes the float 26.103
df = pd.read_csv(INPUT_FILE, dtype={"business_number": str})
print(df.shape)
df.head()

(20, 19)


,business_number,title,business_type,submitter_list,state_list,url,affair_id,submitted_by,councillor_id,parliamentary_group,party,submission_date,submitted_in,state,responsible_authority,first_council,cosigners,cosigner_count,topics
0,26.1035,Kulturgütertransfergesetz– Rechtssicherheit un...,Anfrage,Pamini Paolo,Erledigt,https://www.parlament.ch/de/ratsbetrieb/suche-...,20261035,Pamini Paolo,10842,Fraktion der Schweizerischen Volkspartei,Schweizerische Volkspartei,19.06.2026,Nationalrat,Erledigt,Departement des Innern (EDI),Nationalrat,NaN,0,Internationale Politik; Kultur
1,26.1034,Die USA verhindern AHV-Auszahlungen an Schweiz...,Anfrage,Molina Fabian,Erledigt,https://www.parlament.ch/de/ratsbetrieb/suche-...,20261034,Molina Fabian,4223,Sozialdemokratische Fraktion,Sozialdemokratische Partei der Schweiz,19.06.2026,Nationalrat,Erledigt,Departement des Innern (EDI),Nationalrat,Berli Rudi,1,Internationale Politik; Sozialer Schutz; Staat...
2,26.1032,Die Empfehlungen zur Beschleunigung der Verfah...,Anfrage,Farinelli Alex,Erledigt,https://www.parlament.ch/de/ratsbetrieb/suche-...,20261032,Farinelli Alex,4259,FDP-Liberale Fraktion,FDP.Die Liberalen,18.06.2026,Nationalrat,Erledigt,"Departement für Umwelt, Verkehr, Energie und K...",Nationalrat,NaN,0,Raumplanung und Wohnungswesen; Staatspolitik
3,26.1031,Wahl der Auszahlungshäufigkeit bei Transaktion...,Anfrage,Feller Olivier,Erledigt,https://www.parlament.ch/de/ratsbetrieb/suche-...,20261031,Feller Olivier,4077,FDP-Liberale Fraktion,FDP.Die Liberalen,17.06.2026,Nationalrat,Erledigt,"Departement für Umwelt, Verkehr, Energie und K...",Nationalrat,NaN,0,Finanzwesen; Medien und Kommunikation; Staatsp...
4,26.1030,Wegleitung zum Lohnausweis. Streichung der Emp...,Anfrage,Feller Olivier,Erledigt,https://www.parlament.ch/de/ratsbetrieb/suche-...,20261030,Feller Olivier,4077,FDP-Liberale Fraktion,FDP.Die Liberalen,17.06.2026,Nationalrat,Erledigt,Finanzdepartement (EFD),Nationalrat,NaN,0,Beschäftigung und Arbeit; Steuer


## 0. Duplicates
The same business item could appear twice (e.g. if the result list changed while paging).

In [2]:
print("Duplicate business numbers:", df.duplicated(subset="business_number").sum())
df = df.drop_duplicates(subset="business_number", keep="first").reset_index(drop=True)

Duplicate business numbers: 0


## 1. Gaps / missing data
Empty strings and whitespace are converted to `NaN` first, so that they are counted as missing.
- `cosigners` empty → the business simply has no cosigners (not an error) → replaced by `""`.
- `councillor_id` / `party` missing → submitter is not a person (e.g. a committee) or the detail page could not be read.

In [3]:
df = df.replace(r"^\s*$", np.nan, regex=True)

missing = pd.DataFrame({"missing": df.isna().sum(), "percent": (df.isna().mean() * 100).round(1)})
missing[missing["missing"] > 0]

,missing,percent
cosigners,17,85.0


In [4]:
df["cosigners"] = df["cosigners"].fillna("")

# Rows without detail information (detail page failed) cannot be used for the analysis
detail_missing = df["submission_date"].isna()
print("Rows without detail information:", detail_missing.sum())
df = df[~detail_missing].copy()

# Submitters without party (e.g. committees) are kept, but labelled explicitly
df["party"] = df["party"].fillna("No party")
df["parliamentary_group"] = df["parliamentary_group"].fillna("No group")

Rows without detail information: 0


## 2. Datatypes
After reading the CSV, dates are strings and IDs may be floats (because of `NaN`). `business_number` (e.g. `26.1030`) is an identifier, not a number, and is therefore read as string (see loading cell).

In [5]:
df.dtypes

business_number            str
title                      str
business_type              str
submitter_list             str
state_list                 str
url                        str
affair_id                int64
submitted_by               str
councillor_id            int64
parliamentary_group        str
party                      str
submission_date            str
submitted_in               str
state                      str
responsible_authority      str
first_council              str
cosigners                  str
cosigner_count           int64
topics                     str
dtype: object

In [6]:
df["submission_date"] = pd.to_datetime(df["submission_date"], format="%d.%m.%Y", errors="coerce")
df["councillor_id"] = df["councillor_id"].astype("Int64")      # nullable integer
df["affair_id"] = df["affair_id"].astype("int64")
df["cosigner_count"] = df["cosigner_count"].astype("int64")

for col in ["business_type", "submitted_in", "first_council", "state", "party", "parliamentary_group"]:
    df[col] = df[col].astype("category")

df.dtypes

business_number                     str
title                               str
business_type                  category
submitter_list                      str
state_list                          str
url                                 str
affair_id                         int64
submitted_by                        str
councillor_id                     Int64
parliamentary_group            category
party                          category
submission_date          datetime64[us]
submitted_in                   category
state                          category
responsible_authority               str
first_council                  category
cosigners                           str
cosigner_count                    int64
topics                              str
dtype: object

## 3. Expected range
Check categorical values against the expected set and numeric/date values against plausible bounds.

In [7]:
checks = {
    "business_type == Anfrage": df["business_type"].eq("Anfrage"),
    "state == Erledigt": df["state"].eq("Erledigt"),
    "submitted_in in {Nationalrat, Ständerat}": df["submitted_in"].isin(["Nationalrat", "Ständerat"]),
    "submission_date between 1995 and today": df["submission_date"].between("1995-01-01", pd.Timestamp.today()),
    "cosigner_count >= 0": df["cosigner_count"].ge(0),
    "cosigner_count matches cosigners list": df["cosigner_count"].eq(
        df["cosigners"].apply(lambda s: len(s.split("; ")) if s else 0)),
    "list submitter == detail submitter": df["submitter_list"].eq(df["submitted_by"]),
}
range_check = pd.DataFrame({"violations": {name: int((~ok).sum()) for name, ok in checks.items()}})
range_check

,violations
business_type == Anfrage,0
state == Erledigt,0
"submitted_in in {Nationalrat, Ständerat}",0
submission_date between 1995 and today,0
cosigner_count >= 0,0
cosigner_count matches cosigners list,0
list submitter == detail submitter,0


In [8]:
# Rows violating a hard constraint (wrong type/state/date) are removed
valid = (checks["business_type == Anfrage"] & checks["state == Erledigt"]
         & checks["submission_date between 1995 and today"])
print("Removed rows:", (~valid).sum())
df = df[valid].copy()

Removed rows: 0


## 4. Outliers
Numeric columns: `title_length` and `cosigner_count`.
- `title_length`: roughly symmetric → IQR rule (1.5 × IQR).
- `cosigner_count`: most questions have 0 cosigners, so the IQR is 0 and every value > 0 would be an "outlier". Instead, values above the 99th percentile are flagged.

**Treatment:** these are real values (some questions simply have many cosigners or long titles), so they are **not removed**, but flagged in a column `is_outlier` so they can be excluded in the analysis if needed.

In [9]:
df["title_length"] = df["title"].str.len()

def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

df["is_outlier"] = (iqr_outliers(df["title_length"])
                    | (df["cosigner_count"] > df["cosigner_count"].quantile(0.99)))
print(df[["cosigner_count", "title_length"]].describe())
df.loc[df["is_outlier"], ["business_number", "cosigner_count", "title_length"]]

       cosigner_count  title_length
count       20.000000      20.00000
mean         0.850000      80.25000
std          3.133436      20.99342
min          0.000000      48.00000
25%          0.000000      68.00000
50%          0.000000      81.00000
75%          0.000000      91.00000
max         14.000000     118.00000


,business_number,cosigner_count,title_length
13,26.1021,14,75


## 5. Format the dataset
- Drop redundant columns from the result list (`submitter_list`, `state_list`) after the consistency check above.
- Split `topics` into a list and create a **long table** (one row per business item and topic) – needed for the research questions on subject areas.

In [10]:
df = df.drop(columns=["submitter_list", "state_list"])
df["topics"] = df["topics"].fillna("").apply(lambda s: [t.strip() for t in s.split(";") if t.strip()])

topics_long = (df.explode("topics")
                 .rename(columns={"topics": "topic"})
                 .dropna(subset=["topic"])
                 [["business_number", "submission_date", "submitted_in", "councillor_id", "party", "topic"]])
topics_long.head()

,business_number,submission_date,submitted_in,councillor_id,party,topic
0,26.1035,2026-06-19,Nationalrat,10842,Schweizerische Volkspartei,Internationale Politik
0,26.1035,2026-06-19,Nationalrat,10842,Schweizerische Volkspartei,Kultur
1,26.1034,2026-06-19,Nationalrat,4223,Sozialdemokratische Partei der Schweiz,Internationale Politik
1,26.1034,2026-06-19,Nationalrat,4223,Sozialdemokratische Partei der Schweiz,Sozialer Schutz
1,26.1034,2026-06-19,Nationalrat,4223,Sozialdemokratische Partei der Schweiz,Staatspolitik


## 6. Enrichment
- `submission_year`: for time-based analyses
- `party_short`: short party name (e.g. "SP", "SVP") for readable plots
- `topic_count`: number of subject areas per business item

In [11]:
PARTY_SHORT = {
    "Schweizerische Volkspartei": "SVP",
    "Sozialdemokratische Partei": "SP",
    "FDP": "FDP",
    "Die Mitte": "Mitte",
    "Christlichdemokratische Volkspartei": "CVP",
    "Grünliberale": "GLP",
    "GRÜNE": "GRÜNE",
    "Bürgerlich-Demokratische Partei": "BDP",
    "Evangelische Volkspartei": "EVP",
    "Eidgenössisch-Demokratische Union": "EDU",
    "Lega": "Lega",
    "Mouvement Citoyens": "MCG",
}

def shorten_party(name):
    for key, short in PARTY_SHORT.items():
        if key.lower() in str(name).lower():
            return short
    return "Other"

df["submission_year"] = df["submission_date"].dt.year
df["party_short"] = df["party"].apply(shorten_party).astype("category")
df["topic_count"] = df["topics"].apply(len)

df[["business_number", "party", "party_short", "submission_year", "topic_count"]].head()

,business_number,party,party_short,submission_year,topic_count
0,26.1035,Schweizerische Volkspartei,SVP,2026,2
1,26.1034,Sozialdemokratische Partei der Schweiz,SP,2026,3
2,26.1032,FDP.Die Liberalen,FDP,2026,2
3,26.1031,FDP.Die Liberalen,FDP,2026,4
4,26.1030,FDP.Die Liberalen,FDP,2026,2


## Save

In [12]:
df_out = df.assign(topics=df["topics"].apply("; ".join))
df_out.to_csv(OUTPUT_DIR / "anfragen_clean.csv", index=False, encoding="utf-8-sig")
topics_long.to_csv(OUTPUT_DIR / "anfragen_topics_long.csv", index=False, encoding="utf-8-sig")
print(df_out.shape, topics_long.shape)

(20, 22) (43, 6)
